# 02 Train: YOLO-seg บน WeedyRice (Phase 3)

- ต้องใช้ **GPU** (PC: RTX 2080 SUPER, หรือ Studio Lab GPU runtime) และ kernel **`Python (rwd)`**
- ต้องรัน `01_data_setup.ipynb` ให้ได้ `data/weedyrice_seg` ก่อน
- ค่าทั้งหมด (โมเดล, imgsz, epochs, batch) อยู่ใน `configs/train.yaml` แก้ที่นั่นที่เดียว
- **เทรนต่อได้**: cell ข้อ 3 รันซ้ำได้เสมอ ถ้าเคยเทรนค้างไว้จะเทรนต่อจาก checkpoint ล่าสุด ถ้าเทรนจบแล้วจะไม่ทำอะไร
- วัดผลบน **val** เท่านั้น ใช้เลือกโมเดล ยังไม่ใช่ผลที่เผยแพร่ได้ **test ใช้ครั้งเดียวใน Phase 4**
- `runs/`, `bundles/`, `*.pt` ไม่เข้า git

## 1. Setup

In [ ]:
import os, sys, json, csv, subprocess
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
CONFIG = ROOT / "configs/train.yaml"
PY = sys.executable  # the kernel's python; "!python" would use a different env
print("repo:", ROOT)
print("python:", PY, sys.version.split()[0])
if ".venv" not in PY:
    print("WARNING: kernel is not 'Python (rwd)'. Run scripts/setup_env.sh, then Kernel -> Change Kernel.")

In [ ]:
# ติดตั้ง/อัปเดตแพ็กเกจลง .venv เท่านั้น (ครั้งแรกจะโหลด PyTorch + CUDA หลาย GB)
!"{PY}" -m pip install -q -r requirements.txt

In [ ]:
from rwd.train import load_train_config, run_state

cfg = load_train_config(CONFIG)
RUN = Path(cfg["project"]) / cfg["name"]
LOG = ROOT / "train.log"
print("model:", cfg["model"], "| imgsz:", cfg["imgsz"], "| epochs:", cfg["epochs"], "| batch:", cfg["batch"])
print("run folder:", RUN)
print("state:", run_state(RUN), "(new = ยังไม่เคยเทรน, resumable = เทรนค้าง, finished = เทรนจบแล้ว)")

In [ ]:
# เช็ก GPU, เวอร์ชัน และชุดข้อมูล ต้องเห็น "cuda_available": true และชื่อ GPU
!"{PY}" -m rwd.train --config "{CONFIG}" --check

## 2. Smoke test (ไม่บังคับ)

เทรน 1 epoch บน 10% ของ train ใช้เวลาไม่ถึง 1 นาที เพื่อเช็กว่า pipeline ทำงานและหน่วยความจำพอ
ตัวเลขจาก smoke test ไม่มีความหมาย ข้ามได้ถ้าเคยผ่านแล้ว

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    !"{PY}" -m rwd.train --config "{CONFIG}" --smoke

## 3. Train / Continue

**รัน cell นี้ซ้ำได้เสมอ** มันเลือกเองว่าจะเริ่มใหม่ เทรนต่อ หรือข้าม (ดู `state` ด้านบน)

- `BACKGROUND = True` (แนะนำ): เทรนเป็น process แยก ปิด VS Code/เบราว์เซอร์ได้ การเทรนไม่หยุด ดูความคืบหน้าที่ข้อ 4
- `BACKGROUND = False`: เห็น log ใน cell นี้ แต่ถ้า kernel หลุด การเทรนจะหยุด (รัน cell นี้ใหม่เพื่อเทรนต่อ)
- ถ้า WSL ถูกปิดหรือเครื่องดับ: เปิดใหม่แล้วรัน cell นี้อีกครั้ง จะต่อจาก checkpoint ล่าสุด
- อยากเทรนรอบใหม่ (เช่นเปลี่ยนโมเดล): เปลี่ยน `name` ใน `configs/train.yaml`

In [ ]:
BACKGROUND = True

def training_pids():
    out = subprocess.run(["pgrep", "-f", "rwd.train --config"], capture_output=True, text=True).stdout.split()
    return [p for p in out if p != str(os.getpid())]

if training_pids():
    print("การเทรนกำลังรันอยู่แล้ว (pid", training_pids()[0] + ") ไม่เริ่มซ้ำ ดูความคืบหน้าที่ข้อ 4")
elif BACKGROUND:
    with open(LOG, "a") as f:
        proc = subprocess.Popen([PY, "-m", "rwd.train", "--config", str(CONFIG), "--auto"],
                                stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
    print("started in background, pid", proc.pid, "| log:", LOG)
else:
    !"{PY}" -m rwd.train --config "{CONFIG}" --auto

## 4. Progress

รันซ้ำได้ระหว่างเทรน กราฟมาจาก `results.csv` (ค่าบน val ทุก epoch)

### ดู log แบบ realtime

**ใน Terminal** (กด `Ctrl+C` เพื่อออก การเทรนไม่หยุด):
```bash
# log เต็ม วิ่งตามการเทรนจริง
tail -f ~/rice-weed-detection/train.log

# สรุปทีละ epoch: epoch, เวลา(วินาที), precision, recall, mAP50, mAP50-95 ของ mask บน val
watch -n 10 "tail -n 5 ~/rice-weed-detection/runs/<name>/results.csv | cut -d, -f1,2,12-15"

# การใช้ GPU
watch -n 2 nvidia-smi
```
`<name>` คือค่า `name` ใน `configs/train.yaml` (ดู `run folder` ที่ข้อ 1)

**ใน notebook**: cell "Live view" ด้านล่างจะรีเฟรชเองทุก 10 วินาที กดปุ่ม Stop (Interrupt) ของ cell เพื่อหยุดดู การเทรนไม่หยุด
ถ้าตั้ง `BACKGROUND = False` ที่ข้อ 3 log จะวิ่งใน cell นั้นเอง แต่การเทรนจะหยุดถ้า kernel หลุด

In [ ]:
import matplotlib.pyplot as plt

print("state:", run_state(RUN), "| running:", bool(training_pids()))
res = RUN / "results.csv"
if not res.exists():
    print("ยังไม่มี results.csv (ยังไม่จบ epoch แรก)")
else:
    rows = [{k.strip(): float(v) for k, v in r.items()} for r in csv.DictReader(open(res))]
    ep = [r["epoch"] for r in rows]
    suffix = "(M)" if any(k.endswith("(M)") for k in rows[0]) else "(B)"  # M = mask, B = box
    print(f"epoch {int(ep[-1])}/{cfg['epochs']}  เวลาที่ใช้ {rows[-1]['time'] / 60:.0f} นาที")
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for k in rows[0]:
        if k.startswith("train/") and k.endswith("_loss"):
            axes[0].plot(ep, [r[k] for r in rows], label=k)
        if k.startswith("val/") and k.endswith("_loss"):
            axes[1].plot(ep, [r[k] for r in rows], label=k)
    for name in ("precision", "recall", "mAP50", "mAP50-95"):
        axes[2].plot(ep, [r[f"metrics/{name}{suffix}"] for r in rows], label=f"{name}{suffix}")
    for ax, title in zip(axes, ("train loss", "val loss", "val metrics (not test)")):
        ax.set_title(title); ax.set_xlabel("epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    axes[2].set_ylim(0, 1)
    plt.tight_layout(); plt.show()

In [ ]:
# บรรทัดท้ายของ log (เฉพาะตอนเทรนแบบ background)
if LOG.exists():
    lines = LOG.read_text(errors="replace").replace("\r", "\n").splitlines()
    lines = [l for l in lines if l.strip() and "it/s" not in l and "s/it" not in l]  # ตัดบรรทัด progress bar
    print("\n".join(l[:160] for l in lines[-8:]))

In [ ]:
# Live view: ตารางค่า val ของ epoch ล่าสุด รีเฟรชทุก 10 วินาที จนกว่าจะเทรนจบหรือกด Stop
import time
from IPython.display import clear_output

COLS = ["epoch", "time", "metrics/precision(M)", "metrics/recall(M)", "metrics/mAP50(M)", "metrics/mAP50-95(M)"]
try:
    while True:
        clear_output(wait=True)
        running = bool(training_pids())
        print("state:", run_state(RUN), "| running:", running, "|", time.strftime("%H:%M:%S"))
        if res.exists():
            rows = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(open(res))]
            cols = [c for c in COLS if c in rows[0]] or list(rows[0])[:6]  # box-only models have no (M) columns
            print("  ".join(f"{c.split('/')[-1]:>14}" for c in cols))
            for r in rows[-10:]:
                print("  ".join(f"{float(r[c]):>14.4g}" for c in cols))
        if not running:
            print("ไม่มีการเทรนรันอยู่ (จบแล้วหรือยังไม่เริ่ม)")
            break
        time.sleep(10)
except KeyboardInterrupt:
    print("หยุดดูแล้ว การเทรนยังรันต่อ")


## 5. Result (หลังเทรนจบ)

`metrics.json` เก็บชื่อ GPU, VRAM, เวอร์ชัน, commit และค่าบน **val**
ตัวเลขนี้ใช้เลือกโมเดลเท่านั้น ห้ามนำไปเผยแพร่ ผลจริงต้องมาจาก test ใน Phase 4 และต้องรายงาน recall คู่กันเสมอ

In [ ]:
m = RUN / "metrics.json"
if not m.exists():
    print("ยังเทรนไม่จบ (ยังไม่มี metrics.json)")
else:
    data = json.loads(m.read_text())
    print(data["note"])
    print("GPU:", data["env"]["gpu"], "| VRAM:", data["env"]["vram_gb"], "GB | torch:", data["env"]["torch"],
          "| ultralytics:", data["env"]["ultralytics"], "| commit:", (data["env"]["commit"] or "")[:7])
    for k, v in data["val"].items():
        print(f"  val {k:<26} {v}")
    bundle = ROOT / "bundles" / f"{RUN.name}.tar.gz"
    print("bundle:", bundle, "(มีแล้ว)" if bundle.exists() else "(ยังไม่มี)")

In [ ]:
# ภาพจาก Ultralytics: กราฟรวม และตัวอย่างการทำนายบน val (ซ้าย = label, ขวา = ทำนาย)
from PIL import Image

for name in ("results.png", "val_batch0_labels.jpg", "val_batch0_pred.jpg"):
    p = RUN / name
    if p.exists():
        plt.figure(figsize=(16, 9)); plt.imshow(Image.open(p)); plt.axis("off"); plt.title(name); plt.show()

## Next

- ส่งผลข้อ 5 ให้ Claude แล้วตัดสินใจว่าจะเทรนเพิ่ม ปรับ config หรือไป Phase 4
- Phase 4 (วัดผลบน test ครั้งเดียว, ผลแยกตามความสูงบิน) จะอยู่ใน notebook แยก และต้องได้ OK ก่อน